In [ ]:
#r "FreeXNSE.dll"
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using FreeXNSE;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

In [ ]:
string proj = "ContactLineSingularity";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [ ]:
var sessions = BoSSSshell.wmg.Sessions;
sessions

In [ ]:
//sessions.Take(1).ForEach(s => s.Export().To(Path.GetFullPath(Path.Combine(".", s.ProjectName, s.Name))).WithShadowFields().WithSupersampling(3).Do());

In [ ]:
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.PrintingNip;
using MathNet.Numerics.RootFinding;

In [ ]:
static Func<double, double>[] ExtractSplines(ISessionInfo s){
    var P = s.Timesteps.Last().Fields.Where(f => f.Identification == "Pressure").Single();
    var Ux = s.Timesteps.Last().Fields.Where(f => f.Identification == "VelocityX").Single();
    var Uy = s.Timesteps.Last().Fields.Where(f => f.Identification == "VelocityY").Single();
    var LS = s.Timesteps.Last().Fields.Where(f => f.Identification == "Phi").Single();

    EdgeMask em = new EdgeMask(LS.Basis.GridDat, X => Math.Abs(X[1]) < 1e-12); // lower wall
    var lSspline = Postprocessing.SplineOnEdge(em, LS, 0, out _, out _);
    var pSpline = Postprocessing.SplineOnEdge(em, ((XDGField)P).GetSpeciesShadowField("A"), 0, out _, out _);
    var uxSpline = Postprocessing.SplineOnEdge(em, ((XDGField)Ux).GetSpeciesShadowField("A"), 0, out _, out _);
    var uySpline = Postprocessing.SplineOnEdge(em, ((XDGField)Uy).GetSpeciesShadowField("A"), 0, out _, out _);

    Func<double, double>[] ret = new Func<double, double>[5];
    ret[0] = x => pSpline.Interpolate(x);
    ret[1] = x => uxSpline.Interpolate(x);
    ret[2] = x => uySpline.Interpolate(x);
    ret[3] = x => lSspline.Interpolate(x);


    return ret;
}  

void ExportData(string filename, double[] x, double[] y){
    using(var stw = new StringWriter()) {   

        int n = x.Length;
        for(int i = 0; i < n; i++){
            stw.WriteLine($"{x[i].ToString()} {y[i].ToString()}");
        }
        
        Directory.CreateDirectory(BoSSSshell.wmg.CurrentProject);
        File.WriteAllText(Path.Combine(".",BoSSSshell.wmg.CurrentProject, filename), stw.ToString());
    }
    
}

Fields plotted over the lower wall

In [ ]:
int N = 10000;
var spline = ExtractSplines(sessions.Where(s => s.Name == "StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip0.0_1.0_Visc1.0_-2.0_CAeq0.0000").Single());
var x = GenericBlas.Linspace(-10, 10, N);

// double[,][][] X = new double[2,2][][];
// double[,][][] Y = new double[2,2][][];

// for(int i = 0; i < 2; i++){
//     for(int j = 0; j < 2; j++){
//         X[i,j] = new double[3][];
//         Y[i,j] = new double[3][];
//         for(int k = 0; k < 2; k++){
//             X[i,j][k] = x;
//             Y[i,j][k] = x.Select(xx => spline[i,j][k](xx)).ToArray();
//             //ExportData($"Angle{i}Slip{j}Field{k}.txt", X[i,j][k], Y[i,j][k]);
//         }            
//     }
// }

Plot(x, x.Select(xx => spline[0](xx)).ToArray(), "Pressure").Display();
Plot(x, x.Select(xx => spline[1](xx)).ToArray(), "VelocityX").Display();
Plot(x, x.Select(xx => spline[2](xx)).ToArray(), "VelocityY").Display();


Utility routines to extract the field values over the lower wall as the contact line is approached.  
In both routines the $y(x)$ values are shifted such that $\hat{x}$ denotes the distance from the contact line along the lower wall, and $x_0$ is the contact line position in original coordinates.  
Note how the `GetValuesLog` augments the $y$-values such that they are normalized by the far-field value: $\hat{y}(\hat{x}) = y(x_0-\hat{x})-y(0)$  
Note how the `GetValuesLin` augments the $y$-values such that they are normalized by the contact line value: $\hat{y}(\hat{x}) = y(x_0-\hat{x})-y(x_0)$

In [ ]:
(double[], double[]) GetValuesLin(Func<double, double>[] f, int field){    
    double x0 = Bisection.FindRoot(t => f[3](t), 0, 10);
    
    var x = GenericBlas.Logspace(2.622*1e-7, x0/2, 1000);   // cellsize/10     
    var y = x.Select(xx => Math.Abs(f[field](x0-xx)-f[field](x0))).ToArray();
    return (x,y);
}
(double[], double[]) GetValuesLog(Func<double, double>[] f, int field){    
    double x0 = Bisection.FindRoot(t => f[3](t), 0, 10);
    
    var x = GenericBlas.Logspace(2.622*1e-7, x0/2, 1000);   // cellsize/10     
    var y = x.Select(xx => Math.Abs(f[field](x0-xx)-f[field](0))).ToArray();
    return (x,y);
}

Creation of datafiles for linear (or semilog) and loglog plots as the contactline is approached.  
Export is omitted in this testfile, instead the slopes are compared to static control values.

In [ ]:
Dictionary<string, double[]> EvaluatedSlopes = new Dictionary<string, double[]>();

foreach(var s in sessions){
    var sp = ExtractSplines(s);
    EvaluatedSlopes[s.Name] = new double[3];
    for(int field = 0; field < 3; field++){
        var f = GetValuesLin(sp, field);
        //ExportData($"{s.Name}Field{field}_Lin.txt", f.Item1, f.Item2);
        var g = GetValuesLog(sp, field);
        //ExportData($"{s.Name}Field{field}_Log.txt", g.Item1, g.Item2);
        EvaluatedSlopes[s.Name][field] = ilPSP.DoubleExtensions.LogLogRegressionSlope(g.Item1, g.Item2);
    }
}

In [ ]:
EvaluatedSlopes

These are hardcoded values, see also the Dissertation of Matthias Rieckmann.  
We check that the computed slopes in the fields as the contact line is approached match those at that time up to a threshold of $0.1$

In [3]:
Dictionary<string, double[]> ControlSlopes = new Dictionary<string, double[]>();

ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip1.0_1.0_Visc1.0_-1.0_CAeq0.0000"] = new double[] { -1.0338258344531719, -0.06208881441454055, -0.00021445171077868078 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip1.0_1.0_Visc1.0_-2.0_CAeq0.0000"] = new double[] { -1.033803611523497, -0.05453781395447296, 0.06105703481674911 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip0.0_1.0_Visc1.0_-1.0_CAeq0.0000"] = new double[] { -1.0338426099285127, -0.06208881707218795, -9.05873414096996E-05 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip0.0_1.0_Visc1.0_-2.0_CAeq0.0000"] = new double[] { -1.0321228364967518, -0.05453406287055777, 0.0610323802574858 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip1.0_1.0_Visc1.0_-1.0_CAeq0.0000"] = new double[] { -1.0623410444851056, -0.06427427932985445, -0.000690243436874846 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip1.0_1.0_Visc1.0_-2.0_CAeq0.0000"] = new double[] { -1.0602412142770783, -0.05129646555389476, 0.032647505628416895 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip0.0_1.0_Visc1.0_-1.0_CAeq0.0000"] = new double[] { -1.0623960682881413, -0.06427427984555541, -0.0006322738061034897 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip0.0_1.0_Visc1.0_-2.0_CAeq0.0000"] = new double[] { -1.0596829789071458, -0.051292728656899496, 0.03229616950918817 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_-1.0_LineSlip1.0_1.0_Visc1.0_-1.0_CAeq0.0000"] = new double[] { -1.2980255064690662, -0.0875824281943848, -0.008913650085030799 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_-1.0_LineSlip1.0_1.0_Visc1.0_-2.0_CAeq0.0000"] = new double[] { -1.2440426103617346, -0.05228062112330619, 0.005524217797064391 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_-1.0_LineSlip0.0_1.0_Visc1.0_-1.0_CAeq0.0000"] = new double[] { -1.2980467710250783, -0.08758242765258348, -0.009813069795708434 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_-1.0_LineSlip0.0_1.0_Visc1.0_-2.0_CAeq0.0000"] = new double[] { -1.2439972355535343, -0.052282914240636615, 0.00424185534835285 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip1.0_1.0_Visc1.0_0.0_CAeq0.0000"] = new double[] { -1.0172660094648, -0.21073852512628222, -0.40006789973191814 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip0.0_1.0_Visc1.0_0.0_CAeq0.0000"] = new double[] { -1.0172633547807417, -0.21073853906515003, -0.4004267703768086 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip1.0_1.0_Visc1.0_0.0_CAeq0.0000"] = new double[] { -1.0664200090922442, -0.2483979115106672, -0.41105926562827455 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_0.0_LineSlip0.0_1.0_Visc1.0_0.0_CAeq0.0000"] = new double[] { -1.0664165268427084, -0.2483979294837503, -0.41142416820350575 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_-1.0_LineSlip1.0_1.0_Visc1.0_0.0_CAeq0.0000"] = new double[] { -1.951574225919906, -0.9366646418363954, -0.8306434054286517 };
ControlSlopes["StaticDroplet_DynamicContactAngle_BndySlip1.0_-1.0_LineSlip0.0_1.0_Visc1.0_0.0_CAeq0.0000"] = new double[] { -1.9596596633282515, -0.9377882247580045, -0.8432209933357033 };
ControlSlopes["StaticDroplet_FixedInterface_BndySlip1.0_2.0_Visc1.0_0.0"] = new double[] { -0.00899752405692274, -0.021844698576677177, 0.7378699921657499 };
ControlSlopes["StaticDroplet_FixedInterface_BndySlip1.0_1.0_Visc1.0_0.0"] = new double[] { -0.019692745505598932, -0.023475655579126727, 0.46984617631622805 };
ControlSlopes["StaticDroplet_FixedInterface_BndySlip1.0_0.0_Visc1.0_0.0"] = new double[] { -0.16007901553331452, -0.0389162719852479, 0.48456421383503095 };
ControlSlopes["StaticDroplet_FixedInterface_BndySlip1.0_-1.0_Visc1.0_0.0"] = new double[] { -0.9595558067212729, -0.08825874095350658, -0.5570925498290341 };

In [ ]:
bool success = true;
double thrsh = 0.1;

foreach(var kvp in EvaluatedSlopes){
    double[] eVals = kvp.Value;
    double[] cVals = ControlSlopes[kvp.Key];

    for(int i = 0; i < eVals.Length; i++){
        if(eVals[i] < cVals[i] - thrsh || eVals[i] > cVals[i] + thrsh){
            success &= false;
            Console.WriteLine("Mismatch in computed slopes in {0}, for field {1}", kvp.Value, i);
        }
    }
}

if(!success){
    throw new ApplicationException("Not all slopes match!");
}



Preview of loglog plots as the contact line is approached

In [ ]:
var sess = sessions.Where(s => s.Name == "StaticDroplet_DynamicContactAngle_BndySlip1.0_1.0_LineSlip1.0_1.0_Visc1.0_-1.0_CAeq0.0000").Single();
sess.Name.Display();
var spline = ExtractSplines(sess);
var f = GetValuesLin(spline, 0);
var g = GetValuesLog(spline, 0);

Gnuplot plt = new Gnuplot();
//plt.PlotLogXY(f.Item1, f.Item2, title : "Lin", format: new PlotFormat("-k"));
plt.PlotLogXLogY(g.Item1, g.Item2, title : "Log", format: new PlotFormat("--k"));

plt.Cmd("set grid");
plt.PlotNow().Display();

In [ ]:
ilPSP.DoubleExtensions.LogLogRegressionSlope(g.Item1, g.Item2).Display()